# YOLOv9 Knowledge Distillation with LibreYOLO

This tutorial walks through the MGD (Masked Generative Distillation, ECCV 2022) and CWD (Channel-Wise Distillation, ICCV 2021) distillation support added to LibreYOLO (branch `41-research-mgd-and-cwd-distillation`).

You'll learn how to:
1. Load a frozen teacher and a randomly-initialized student.
2. Train the student with distillation loss layered on top of the detection loss.
3. Swap between MGD and CWD with a single argument.
4. Publish the distilled student to the Hugging Face Hub.

The notebook runs on CPU in ~30 seconds because we use `imgsz=128` and a tiny synthetic dataset. The same code, with real weights and COCO, produces a genuinely distilled small detector.

---

## 1. Install & imports

In [ ]:
# pip install -e git+https://github.com/aalvsz/libreyolo@agentic/b-distillation#egg=libreyolo
from pathlib import Path
import numpy as np
import torch
import yaml
from PIL import Image

from libreyolo.models.yolo9.model import LibreYOLO9
from libreyolo.models.yolo9.nn import LibreYOLO9Model
from libreyolo.distillation import Distiller

torch.manual_seed(0)

## 2. Make a tiny detection dataset

Standard YOLO format: `class cx cy w h`, normalized. For real runs, point `data.yaml` at your COCO (or VOC, or VisDrone) directory.

In [ ]:
DATA = Path('demo_distill').resolve()
(DATA / 'images/train').mkdir(parents=True, exist_ok=True)
(DATA / 'images/val').mkdir(parents=True, exist_ok=True)
(DATA / 'labels/train').mkdir(parents=True, exist_ok=True)
(DATA / 'labels/val').mkdir(parents=True, exist_ok=True)

def sample(split, i, cls=0):
    rng = np.random.default_rng(i)
    img = rng.integers(60, 110, size=(128, 128, 3), dtype=np.uint8)
    img[24:64, 24:64] = 220
    Image.fromarray(img).save(DATA / 'images' / split / f'{i}.jpg', quality=80)
    (DATA / 'labels' / split / f'{i}.txt').write_text(f'{cls} 0.34375 0.34375 0.3125 0.3125\n')

for i in range(8): sample('train', i)
for i in range(2): sample('val', i)

(DATA / 'data.yaml').write_text(yaml.dump({
    'path': str(DATA), 'train': 'images/train', 'val': 'images/val',
    'nc': 1, 'names': ['square'],
}))
print('dataset ready')

## 3. Build teacher + student

Distillation in LibreYOLO works on any two models from the same family. The teacher must be loadable by `LibreYOLO(path)` (the unified factory auto-detects family/size from the state dict).

For a real experiment: use `LibreYOLO9c.pt` as teacher and `LibreYOLO9t.pt` as student. Here we make fresh (random) checkpoints so the notebook is offline.

In [ ]:
teacher_ckpt = DATA / 'teacher-c.pt'
student_ckpt = DATA / 'student-t.pt'
torch.save({'model': LibreYOLO9Model(config='c', nb_classes=1).state_dict()}, teacher_ckpt)
torch.save({'model': LibreYOLO9Model(config='t', nb_classes=1).state_dict()}, student_ckpt)

student = LibreYOLO9(model_path=str(student_ckpt), size='t', nb_classes=1, device='cpu')
print(f'student params: {sum(p.numel() for p in student.model.parameters()):,}')

## 4. Train with MGD distillation

The `train()` API forwards distillation kwargs straight through to the trainer. Any of `distill=True`, `distill_teacher`, `distill_loss_type`, `distill_loss_weight`, `distill_mask_ratio` (MGD only), `distill_tau` (CWD only) can be used.

In [ ]:
results_mgd = student.train(
    data=str(DATA / 'data.yaml'),
    epochs=2, batch=2, imgsz=128, lr0=0.001, optimizer='SGD',
    device='cpu', workers=0, amp=False, patience=2,
    project=str(DATA / 'runs'), name='mgd', exist_ok=True,
    distill=True,
    distill_teacher=str(teacher_ckpt),
    distill_loss_type='mgd',
    distill_loss_weight=0.5,
    distill_mask_ratio=0.65,
)
print(f"MGD   final_loss = {results_mgd['final_loss']:.4f}   best_ckpt = {results_mgd['best_checkpoint']}")

## 5. Switch to CWD with a single kwarg

CWD (Channel-Wise Distillation) matches per-channel softmax distributions between teacher and student features. It's often a strong baseline for detection — easy to try.

In [ ]:
# Re-init student (otherwise we'd continue from the MGD-trained weights)
student2 = LibreYOLO9(model_path=str(student_ckpt), size='t', nb_classes=1, device='cpu')

results_cwd = student2.train(
    data=str(DATA / 'data.yaml'),
    epochs=2, batch=2, imgsz=128, lr0=0.001, optimizer='SGD',
    device='cpu', workers=0, amp=False, patience=2,
    project=str(DATA / 'runs'), name='cwd', exist_ok=True,
    distill=True,
    distill_teacher=str(teacher_ckpt),
    distill_loss_type='cwd',
    distill_loss_weight=0.5,
    distill_tau=1.0,
)
print(f"CWD   final_loss = {results_cwd['final_loss']:.4f}   best_ckpt = {results_cwd['best_checkpoint']}")

## 6. What's actually happening

The `Distiller` module is architecture-agnostic. It:
1. Registers forward hooks on each model at configured tap points (backbone / neck outputs).
2. Runs a frozen teacher forward on every batch.
3. During the student forward (the normal training step), the hooks capture matching student features.
4. `compute_loss()` then runs the chosen distillation loss (MGD or CWD) at each scale, sums them, and applies the user's weight.
5. This total is added to the student's detection loss and backpropagated.

Tap points and channel dims come from each model's `get_distill_config()` — so adding another architecture (YOLOX, RF-DETR, future YOLO-NAS) just means implementing that method on the wrapper.

### Direct use of `Distiller`

If you want to drive the loop manually (for custom training loops), the primitive is:

In [ ]:
teacher_wrap = LibreYOLO9(model_path=str(teacher_ckpt), size='c', nb_classes=1, device='cpu')
student_wrap = LibreYOLO9(model_path=str(student_ckpt), size='t', nb_classes=1, device='cpu')

dist = Distiller(
    teacher_model=teacher_wrap.model,
    student_model=student_wrap.model,
    teacher_config=teacher_wrap.get_distill_config(),
    student_config=student_wrap.get_distill_config(),
    loss_type='mgd', loss_weight=0.5,
)

x = torch.randn(1, 3, 128, 128)
dist.teacher_forward(x)
_ = student_wrap.model(x)  # student forward populates hooks
loss = dist.compute_loss()
print('distillation loss (scalar tensor):', loss.item())
dist.step()  # clear hook state

## 7. Train at scale + push to Hub

For a real experiment — e.g., YOLOv9-c → YOLOv9-t on COCO — use the script:

```bash
python scripts/train_distill_yolo9.py \
    --data coco/data.yaml \
    --teacher weights/LibreYOLO9c.pt \
    --student-size t \
    --init-student weights/LibreYOLO9t.pt \
    --epochs 100 --batch 16 --imgsz 640 \
    --loss-type mgd --loss-weight 0.5 \
    --push --hf-repo ander2221/libreyolo-yolo9t-distilled
```

See `docs/agentic-features/blog/yolo9-distillation.md` for HF Jobs submission, expected wall-clock, and how distilled students compare to the non-distilled baseline.